In [ ]:
import os

import datasets
import polars as pl
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    export_static_quantized_openvino_model,
)

# Monkey patch the dataset loading function
original_load_dataset = datasets.load_dataset


def patched_load_dataset(path, *args, **kwargs):
    if os.path.exists(path) and os.path.isdir(path):
        # If it's a local directory, use load_from_disk
        return datasets.load_from_disk(path)
    else:
        # Otherwise use original function
        return original_load_dataset(path, *args, **kwargs)


# Apply the patch
datasets.load_dataset = patched_load_dataset


posts = (
    pl.read_parquet("data/posts.pq", columns=["Текст"])
    .rename({"Текст": "text"})
    .with_columns(pl.col("text").str.replace_all("_x000D_", "\n").str.strip_chars())
    .sample(256)
)

ds = Dataset.from_polars(posts)
ds.save_to_disk("./data/local_calibration_dataset")

# model export to openvino
model_path = "data/rubert_tiny2_001"
model = SentenceTransformer(model_path, backend="openvino")
export_static_quantized_openvino_model(
    model=model,
    quantization_config=None,
    push_to_hub=False,
    create_pr=False,
    model_name_or_path=model_path,
    dataset_name="./data/local_calibration_dataset",
    column_name="text",  # Column containing text data
    dataset_split="train",  # Use the entire dataset
    dataset_config_name="default",
)

No 'openvino_model.xml' found in 'data/models/rubert-tiny2'. Exporting the model to OpenVINO.


/home/sigma.sbrf.ru@16525722/projects/rag-files-back/.venv/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask
Saving the exported OpenVINO model is heavily recommended to avoid having to export it again. Do so with `model.save_pretrained('data/models/rubert-tiny2')`.


/home/sigma.sbrf.ru@16525722/projects/rag-files-back/.venv/lib/python3.13/site-packages/rich/live.py:256: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/sigma.sbrf.ru@16525722/projects/rag-files-back/.venv/lib/python3.13/site-packages/rich/live.py:256: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [ ]:
model = SentenceTransformer(
    model_path,
    backend="openvino",
    model_kwargs={"file_name": "openvino/openvino_model_qint8_quantized.xml"},
)
model.encode(["привет"])